In [2]:
%pip install azure-search-documents==11.4.0 openai dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Load environment variables from .env file
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from openai import AzureOpenAI
import os

load_dotenv(override=True)

# Azure AI Search
search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
index_name = os.getenv("AZURE_SEARCH_INDEX", "rag-1776332199094")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY") or os.getenv("AZURE_SEARCH_ADMIN_KEY")

# Prefer API key to avoid RBAC issues during local notebooks.
if search_api_key:
    credential = AzureKeyCredential(search_api_key)
    search_auth_mode = "API key"
else:
    credential = DefaultAzureCredential()
    search_auth_mode = "DefaultAzureCredential (RBAC)"

# Azure OpenAI (embeddings)
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
if not azure_openai_endpoint:
    # Optional fallback if you have Foundry-style env vars
    foundry_base = os.getenv("AZURE_FOUNDRY_OPENAI_BASE_URL", "")
    if "/openai" in foundry_base:
        azure_openai_endpoint = foundry_base.split("/openai")[0]

azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")

if not azure_openai_endpoint:
    raise ValueError("Missing AZURE_OPENAI_ENDPOINT in .env")
if not azure_openai_api_key:
    raise ValueError("Missing AZURE_OPENAI_API_KEY in .env")

client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_api_key,
    api_version="2024-02-01"
 )

print(f"Using Azure Search endpoint: {search_endpoint}")
print(f"Using Azure Search index: {index_name}")
print(f"Azure Search auth mode: {search_auth_mode}")
print(f"Using embedding deployment: {embedding_deployment}")

Using Azure Search endpoint: https://ai-search-mario2.search.windows.net
Using Azure Search index: rag-1776332199094
Azure Search auth mode: API key
Using embedding deployment: text-embedding-ada-002






## PARTE 2 — Búsquedas prácticas (ejecución en Python)

Entregar un script o notebook (.ipynb recomendado) con ejecuciones de los siguientes tipos de búsqueda y ejemplos de resultados (top-5):

1. Vector Search
2. Hybrid Search (vector + keyword)
3. Semantic Search (query_type=semantic)
4. Semantic Hybrid Search (semantic + vector)

[Referencia para semantic ranking](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/Quickstart-Semantic-Ranking/semantic-ranking-quickstart.ipynb)
[Referencia para vector search](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/Quickstart-Vector-Search/vector-search-quickstart.ipynb)


### Vector Search

In [4]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from textwrap import fill, indent

# Inicializamos el cliente
search_client = SearchClient(search_endpoint, index_name, credential)
query = "¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?"

# 1) Generar vector con Azure OpenAI
response = client.embeddings.create(
    input=query,
    model=embedding_deployment
)
embedding = response.data[0].embedding

# 2) Construir la query vectorial
vector_query = VectorizedQuery(
    vector=embedding,
    k_nearest_neighbors=5,
    fields="text_vector"  # Asegúrate de que este es el nombre correcto en tu índice
)

# 3) Ejecutar búsqueda
results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "chunk"],
    top=5
)

rows = list(results)

print("╔══════════════════════════════════════════════════════════════════════════════════════╗")
print("║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║")
print("╚══════════════════════════════════════════════════════════════════════════════════════╝")
print(f" 🎯 Búsqueda: {query}")
print(f" 📂 Índice:   {index_name}")
print(f" 📊 Matches:  {len(rows)} fragmentos recuperados")
print("────────────────────────────────────────────────────────────────────────────────────────")

for i, result in enumerate(rows, start=1):
    title = result.get("title") or "(sin título)"
    chunk = (result.get("chunk") or "").strip()
    score = float(result.get("@search.score", 0))

    # Limpiamos saltos de línea extraños del PDF y limitamos a 450 caracteres
    chunk_clean = " ".join(chunk.split())
    if len(chunk_clean) > 450:
        chunk_clean = chunk_clean[:450] + "..."

    # Formateamos el bloque de texto
    snippet = fill(chunk_clean, width=80)

    print(f" 🔹 RESULTADO #{i}  |  Relevancia (Score): {score:.4f}")
    print(f" 📄 Documento: {title}")
    print(" 💬 Contenido recuperado:")
    # Identamos el fragmento para que se diferencie visualmente de los metadatos
    print(indent(snippet, "      "))
    print("────────────────────────────────────────────────────────────────────────────────────────")

╔══════════════════════════════════════════════════════════════════════════════════════╗
║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
 🎯 Búsqueda: ¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?
 📂 Índice:   rag-1776332199094
 📊 Matches:  5 fragmentos recuperados
────────────────────────────────────────────────────────────────────────────────────────
 🔹 RESULTADO #1  |  Relevancia (Score): 0.8680
 📄 Documento: RAG Champions League Cuartos 2025_2026.pdf
 💬 Contenido recuperado:
      https://www.youtube.com/watch?v=njFX8EKiYrY
      https://www.notimerica.com/deportes/noticia-futbol-champions-cronica-atletico-
      madrid-fc-barcelona-20260414231325.html
      https://www.notimerica.com/deportes/noticia-futbol-champions-cronica-atletico-
      madrid-fc-barcelona-20260414231325.html
      

### Hybrid Search

In [5]:
# HYBRID SEARCH = keyword + vector
from azure.search.documents.models import VectorizedQuery
from textwrap import fill, indent

hybrid_query = "¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?"

# 1) Embedding de la consulta (parte vectorial)
hybrid_embedding = client.embeddings.create(
    input=hybrid_query,
    model=embedding_deployment
).data[0].embedding

vector_query = VectorizedQuery(
    vector=hybrid_embedding,
    k_nearest_neighbors=5,
    fields="text_vector"
)

# 2) Búsqueda híbrida: search_text (keyword) + vector_queries (vector)
hybrid_results = search_client.search(
    search_text=hybrid_query,
    vector_queries=[vector_query],
    select=["title", "chunk"],
    top=5
)

hybrid_rows = list(hybrid_results)
print("╔══════════════════════════════════════════════════════════════════════════════════════╗")
print("║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║")
print("╚══════════════════════════════════════════════════════════════════════════════════════╝")
print(f" 🎯 Búsqueda: {hybrid_query}")
print(f" 📂 Índice:   {index_name}")
print(f" 📊 Matches:  {len(hybrid_rows)} fragmentos recuperados")
print("────────────────────────────────────────────────────────────────────────────────────────")

for i, result in enumerate(hybrid_rows, start=1):
    title = result.get("title") or "(sin título)"
    chunk = (result.get("chunk") or "").strip()
    score = float(result.get("@search.score", 0))

    # Limpiamos saltos de línea extraños del PDF y limitamos a 450 caracteres
    chunk_clean = " ".join(chunk.split())
    if len(chunk_clean) > 450:
        chunk_clean = chunk_clean[:450] + "..."

    # Formateamos el bloque de texto
    snippet = fill(chunk_clean, width=80)

    print(f" 🔹 RESULTADO #{i}  |  Relevancia (Score): {score:.4f}")
    print(f" 📄 Documento: {title}")
    print(" 💬 Contenido recuperado:")
    # Identamos el fragmento para que se diferencie visualmente de los metadatos
    print(indent(snippet, "      "))
    print("────────────────────────────────────────────────────────────────────────────────────────")

╔══════════════════════════════════════════════════════════════════════════════════════╗
║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
 🎯 Búsqueda: ¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?
 📂 Índice:   rag-1776332199094
 📊 Matches:  5 fragmentos recuperados
────────────────────────────────────────────────────────────────────────────────────────
 🔹 RESULTADO #1  |  Relevancia (Score): 0.0325
 📄 Documento: RAG Champions League Cuartos 2025_2026.pdf
 💬 Contenido recuperado:
      a la desesperada por forzar la prórroga, el Bayern armó un contragolpe de manual
      en el cuarto minuto de tiempo de descuento (90+4'). Michael Olise, culminando
      una actuación individual soberbia, definió con frialdad para sellar la victoria
      del Bayern por 4-3, confirmando su merecido pase a semi

### Semantic Search

In [6]:
# SEMANTIC SEARCH = keyword + semantic ranker (sin vector)
from textwrap import fill, indent
from azure.search.documents.indexes import SearchIndexClient
from azure.core.exceptions import HttpResponseError

semantic_query = "¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?"
semantic_config = os.getenv("AZURE_SEARCH_SEMANTIC_CONFIG")

# Si no se define por variable de entorno, intentamos descubrir una configuración válida del índice
if not semantic_config:
    index_client = SearchIndexClient(search_endpoint, credential)
    index_def = index_client.get_index(index_name)
    semantic_search_cfg = getattr(index_def, "semantic_search", None)
    semantic_configs = getattr(semantic_search_cfg, "configurations", []) if semantic_search_cfg else []
    semantic_config = semantic_configs[0].name if semantic_configs else None

search_args = {
    "search_text": semantic_query,
    "select": ["title", "chunk"],
    "top": 5,
}

if semantic_config:
    search_args["query_type"] = "semantic"
    search_args["semantic_configuration_name"] = semantic_config

try:
    semantic_results = search_client.search(**search_args)
except HttpResponseError as ex:
    # Si la config indicada es inválida, reintentamos sin semántica para no romper la práctica
    print(f"Aviso: no se pudo usar semantic config '{semantic_config}': {ex}")
    semantic_config = None
    semantic_results = search_client.search(
        search_text=semantic_query,
        select=["title", "chunk"],
        top=5
    )

semantic_rows = list(semantic_results)

print("╔══════════════════════════════════════════════════════════════════════════════════════╗")
print("║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║")
print("╚══════════════════════════════════════════════════════════════════════════════════════╝")
print(f" 🎯 Búsqueda: {semantic_query}")
print(f" 📂 Índice:   {index_name}")
print(f" 📊 Matches:  {len(semantic_rows)} fragmentos recuperados")
print("────────────────────────────────────────────────────────────────────────────────────────")

for i, result in enumerate(semantic_rows, start=1):
    title = result.get("title") or "(sin título)"
    chunk = (result.get("chunk") or "").strip()
    score = float(result.get("@search.score", 0))

    # Limpiamos saltos de línea extraños del PDF y limitamos a 450 caracteres
    chunk_clean = " ".join(chunk.split())
    if len(chunk_clean) > 450:
        chunk_clean = chunk_clean[:450] + "..."

    # Formateamos el bloque de texto
    snippet = fill(chunk_clean, width=80)

    print(f" 🔹 RESULTADO #{i}  |  Relevancia (Score): {score:.4f}")
    reranker_score = result.get("@search.reranker_score", None)
    if reranker_score is not None:
        print(f"    🧠 Reranker score: {float(reranker_score):.4f}")
    print(f" 📄 Documento: {title}")
    print(" 💬 Contenido recuperado:")
    # Identamos el fragmento para que se diferencie visualmente de los metadatos
    print(indent(snippet, "      "))
    print("────────────────────────────────────────────────────────────────────────────────────────")

╔══════════════════════════════════════════════════════════════════════════════════════╗
║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
 🎯 Búsqueda: ¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?
 📂 Índice:   rag-1776332199094
 📊 Matches:  5 fragmentos recuperados
────────────────────────────────────────────────────────────────────────────────────────
 🔹 RESULTADO #1  |  Relevancia (Score): 8.8582
    🧠 Reranker score: 2.7983
 📄 Documento: RAG Champions League Cuartos 2025_2026.pdf
 💬 Contenido recuperado:
      Para este estudio de caso se ha analizado exhaustivamente la distribución de la
      posesión, la calidad intrínseca de las oportunidades creadas (medidas a través
      del estándar de Goles Esperados - xG), y la influencia, a menudo definitiva, del
      factor del comportamiento

### Semantic Hybrid Search

In [7]:
# SEMANTIC HYBRID SEARCH = keyword + vector + semantic ranker
from azure.search.documents.models import VectorizedQuery
from azure.search.documents.indexes import SearchIndexClient
from azure.core.exceptions import HttpResponseError
from textwrap import fill, indent

semantic_hybrid_query = "¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?"

# 1) Embedding de la consulta (parte vectorial)
semantic_hybrid_embedding = client.embeddings.create(
    input=semantic_hybrid_query,
    model=embedding_deployment
).data[0].embedding

vector_query = VectorizedQuery(
    vector=semantic_hybrid_embedding,
    k_nearest_neighbors=5,
    fields="text_vector"
)

# 2) Resolver semantic config válida (env var o autodetección en el índice)
semantic_config = os.getenv("AZURE_SEARCH_SEMANTIC_CONFIG")
if not semantic_config:
    index_client = SearchIndexClient(search_endpoint, credential)
    index_def = index_client.get_index(index_name)
    semantic_search_cfg = getattr(index_def, "semantic_search", None)
    semantic_configs = getattr(semantic_search_cfg, "configurations", []) if semantic_search_cfg else []
    semantic_config = semantic_configs[0].name if semantic_configs else None

# 3) Ejecutar Semantic Hybrid Search
search_args = {
    "search_text": semantic_hybrid_query,
    "vector_queries": [vector_query],
    "select": ["title", "chunk"],
    "top": 5,
}

if semantic_config:
    search_args["query_type"] = "semantic"
    search_args["semantic_configuration_name"] = semantic_config

try:
    semantic_hybrid_results = search_client.search(**search_args)
except HttpResponseError as ex:
    # Si falla la semantic config, mantenemos la búsqueda híbrida sin semantic ranker
    print(f"Aviso: no se pudo usar semantic config '{semantic_config}': {ex}")
    semantic_config = None
    semantic_hybrid_results = search_client.search(
        search_text=semantic_hybrid_query,
        vector_queries=[vector_query],
        select=["title", "chunk"],
        top=5
    )

semantic_hybrid_rows = list(semantic_hybrid_results)

print("╔══════════════════════════════════════════════════════════════════════════════════════╗")
print("║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║")
print("╚══════════════════════════════════════════════════════════════════════════════════════╝")
print(f" 🎯 Búsqueda: {semantic_hybrid_query}")
print(f" 📂 Índice:   {index_name}")
print(f" 📊 Matches:  {len(semantic_hybrid_rows)} fragmentos recuperados")
print("────────────────────────────────────────────────────────────────────────────────────────")

for i, result in enumerate(semantic_hybrid_rows, start=1):
    title = result.get("title") or "(sin título)"
    chunk = (result.get("chunk") or "").strip()
    score = float(result.get("@search.score", 0))

    # Limpiamos saltos de línea extraños del PDF y limitamos a 450 caracteres
    chunk_clean = " ".join(chunk.split())
    if len(chunk_clean) > 450:
        chunk_clean = chunk_clean[:450] + "..."

    # Formateamos el bloque de texto
    snippet = fill(chunk_clean, width=80)

    print(f" 🔹 RESULTADO #{i}  |  Relevancia (Score): {score:.4f}")
    reranker_score = result.get("@search.reranker_score", None)
    if reranker_score is not None:
        print(f"    🧠 Reranker score: {float(reranker_score):.4f}")
    print(f" 📄 Documento: {title}")
    print(" 💬 Contenido recuperado:")
    # Identamos el fragmento para que se diferencie visualmente de los metadatos
    print(indent(snippet, "      "))
    print("────────────────────────────────────────────────────────────────────────────────────────")

╔══════════════════════════════════════════════════════════════════════════════════════╗
║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
 🎯 Búsqueda: ¿Qué equipo se clasificó y cuál fue el resultado global en la eliminatoria entre FC Barcelona y Atlético de Madrid?
 📂 Índice:   rag-1776332199094
 📊 Matches:  5 fragmentos recuperados
────────────────────────────────────────────────────────────────────────────────────────
 🔹 RESULTADO #1  |  Relevancia (Score): 0.0156
    🧠 Reranker score: 2.7983
 📄 Documento: RAG Champions League Cuartos 2025_2026.pdf
 💬 Contenido recuperado:
      Para este estudio de caso se ha analizado exhaustivamente la distribución de la
      posesión, la calidad intrínseca de las oportunidades creadas (medidas a través
      del estándar de Goles Esperados - xG), y la influencia, a menudo definitiva, del
      factor del comportamiento

## ⭐ Extra (elige al menos UNA opción y documenta)

A. Añadir una **custom skill** (OCR o endpoint propio) al skillset: documentar su implementación y efecto en indexing. Referencia: https://learn.microsoft.com/en-us/azure/search/cognitive-search-defining-skillset

B. Añadir un **scoring profile** al índice y explicar cómo afecta al ranking. Referencia: https://learn.microsoft.com/es-es/azure/search/index-add-scoring-profiles?tabs=python

C. Implementar **multimodal search** (texto + imágenes) si los documentos contienen imágenes. Referencia: https://learn.microsoft.com/en-us/azure/search/multimodal-search-overview

---





## Entregables (qué debe contener la entrega)

1. Un archivo en formato .docx, .ipynb o .md documentando la Parte 1
2. `practica_vector_search.ipynb` o `practica_vector_search.py` con la Parte 2 ejecutada y resultados incluidos.
4. (Opcional) Documentación y explicación de la implementación del extra elegido

---
